In [ ]:
# ============================================
# CELL 1: SETUP KAGGLE ENVIRONMENT
# ============================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

!pip install -q transformers==4.46.3 datasets sentencepiece accelerate peft==0.10.0

# THƯ MỤC ĐẦU RA (Lưu model sau khi train)
WORKING_DIR = "/kaggle/working/Summarization"
os.makedirs(WORKING_DIR, exist_ok=True)

print(f"💾 Thư mục Output (để save model): {WORKING_DIR}")

In [ ]:
# ============================================
# CELL 2: LOAD BASE MODEL & INIT LORA
# ============================================

import torch
import os
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "VietAI/vit5-large"

FINAL_MODEL_OUTPUT = os.path.join(WORKING_DIR, "vit5-summarization-adapter")

print("Đang tải Tokenizer và Base Model (Float16)...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=False)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
)
model.enable_input_require_grads()

print("🌱 Khởi tạo cấu trúc LoRA mới tinh...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, lora_config)

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.print_trainable_parameters()

In [ ]:
# ============================================
# CELL 3: LOAD DATASET
# ============================================

from datasets import load_dataset

print(f"⏳ Đang tải 30.000 mẫu Train của VietNews...")
train_chunk = load_dataset("nam194/vietnews", split="train[:30000]")

print("⏳ Đang tải 10.000 mẫu Validation của VietNews...")
val_chunk = load_dataset("nam194/vietnews", split="validation[:10000]")

def preprocess_summarization(examples):
    clean_articles = [doc.replace('_', ' ') for doc in examples["article"]]
    clean_abstracts = [abs.replace('_', ' ') for abs in examples["abstract"]]

    inputs = [f"tóm tắt: {doc}" for doc in clean_articles]
    targets = clean_abstracts

    model_inputs = tokenizer(inputs, max_length=1024, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=200, padding="max_length", truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("⏳ Đang Tokenize dữ liệu Train...")
tokenized_train = train_chunk.map(preprocess_summarization, batched=True, remove_columns=train_chunk.column_names)

print("⏳ Đang Tokenize dữ liệu Validation...")
tokenized_val = val_chunk.map(preprocess_summarization, batched=True, remove_columns=val_chunk.column_names)

In [ ]:
# ============================================
# CELL 4: TRAIN & SAVE
# ============================================

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

OUTPUT_DIR = os.path.join(WORKING_DIR, "vit5-summarization-lora")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    per_device_eval_batch_size=4,
    optim="adamw_torch",
    fp16=True,
    learning_rate=2e-4,
    num_train_epochs=1,
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=10,
    overwrite_output_dir=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

trainer.train()

# Lưu Adapter hoàn chỉnh ra thư mục /kaggle/working/
trainer.model.save_pretrained(FINAL_MODEL_OUTPUT)